In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
df_test=pd.read_csv('test_engineered.csv')
df_train=pd.read_csv('train_engineered.csv')

In [4]:
from sklearn.model_selection import train_test_split
X=df_train.drop(columns=['demand'])
y=df_train['demand']
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.2,random_state=42)


In [5]:
import optuna
import xgboost as xgb
import numpy as np
from sklearn import metrics

def objective_xgb(trial):
    param = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'random_state': 42,
        'tree_method': 'hist',  # Accelerates training inside Optuna loops
        'n_jobs': -1,
        
        # Search space boundaries
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),  # Deeper trees (>10) overfit traffic data
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'alpha': trial.suggest_float('alpha', 0.1, 15.0, log=True),      # L1 Lasso Regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True) # L2 Ridge Regularization
    }

    # Use the clean, isolated encoded matrices created outside the loop
    model = xgb.XGBRegressor(**param)
    model.fit(X_train_optuna, y_train, verbose=False)

    preds = model.predict(X_val_optuna)
    preds = np.clip(preds, a_min=0, a_max=None)
    
    return metrics.r2_score(y_val, preds)

print("Pre-processing clean, isolated streams for tuning...")
# FIX: Use new distinct variable names so re-running the cell never corrupts your data
# Assuming X_test is your validation fold
# Already-encoded features — don't use traffic_pipeline
X_train_optuna = X_train.copy()
X_val_optuna = X_val.copy()
# y_val is already defined from train_test_split and will be used directly

print("Starting leak-free XGBoost hyperparameter tuning...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30)

print("\n--- XGBOOST TUNING COMPLETE ---")
print(f"Best Tuned Validation R2 Score: {study_xgb.best_value * 100:.2f}%")
print("Best Hyperparameters Found:", study_xgb.best_params)

c:\Users\Pulkit\Banglore_traffic_prediction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-29 12:51:08,051] A new study created in memory with name: no-name-9aaa6f78-6f2e-4bed-9149-40c625944fb9


Pre-processing clean, isolated streams for tuning...
Starting leak-free XGBoost hyperparameter tuning...


[I 2026-05-29 12:51:13,445] Trial 0 finished with value: 0.8293622492788632 and parameters: {'n_estimators': 1200, 'learning_rate': 0.04198187548445219, 'max_depth': 6, 'subsample': 0.7208231010095074, 'colsample_bytree': 0.6377252200135861, 'min_child_weight': 7, 'alpha': 14.06195702004995, 'reg_lambda': 5.061805130066624}. Best is trial 0 with value: 0.8293622492788632.
[I 2026-05-29 12:51:27,348] Trial 1 finished with value: 0.9351017474508916 and parameters: {'n_estimators': 2300, 'learning_rate': 0.05147138696112851, 'max_depth': 10, 'subsample': 0.9791322113086046, 'colsample_bytree': 0.6636812223781324, 'min_child_weight': 2, 'alpha': 0.14114303996132466, 'reg_lambda': 0.3620043374373885}. Best is trial 1 with value: 0.9351017474508916.
[I 2026-05-29 12:51:32,859] Trial 2 finished with value: 0.8847653820258761 and parameters: {'n_estimators': 1400, 'learning_rate': 0.01061400799279587, 'max_depth': 7, 'subsample': 0.7827175774809311, 'colsample_bytree': 0.6819963148080034, 'min


--- XGBOOST TUNING COMPLETE ---
Best Tuned Validation R2 Score: 94.12%
Best Hyperparameters Found: {'n_estimators': 2000, 'learning_rate': 0.12248300110815602, 'max_depth': 9, 'subsample': 0.9937473251935228, 'colsample_bytree': 0.9992960527684035, 'min_child_weight': 1, 'alpha': 0.10153662406634725, 'reg_lambda': 1.7264048068100195}


In [37]:
import xgboost as xgb
import numpy as np
from sklearn import metrics

# 1. Pre-process clean, isolated streams for final training
print("Transforming feature matrices through the traffic pipeline...")


# 2. Extract and combine the best parameters dynamically
print("Extracting winning parameters from Optuna study...")
production_params = {
    'objective': 'reg:squarederror', 
    'eval_metric': 'rmse',
    'tree_method': 'hist',              # Fast histogram binning for large data
    'n_jobs': -1,                       # Use all CPU threads
    'random_state': 42,
    
    'n_estimators': 2000,
  'learning_rate': 0.017801224644221823,
  'max_depth': 15,
  'subsample': 0.7531617977880385,
  'colsample_bytree': 0.9694047886890467,
  'min_child_weight': 2,
    
    # # Optional: Keep these fallback regularizations to protect against lookup overfitting
    # 'alpha': study_xgb.best_params.get('alpha', 5.0),
    # 'reg_lambda': study_xgb.best_params.get('reg_lambda', 10.0)
}

# 3. Initialize the production model
model_xgb = xgb.XGBRegressor(**production_params)

# 4. Train the model on 100% of the training features
print("Fitting final production XGBoost model...")
model_xgb.fit(X_train, y_train)

# 5. Make predictions and safely bound them
print("Generating test inferences...")
predictions = model_xgb.predict(X_val)
predictions = np.clip(predictions, a_min=0, a_max=None) # Ground negative predictions to 0

# 6. Evaluate final R2 score performance
final_r2 = metrics.r2_score(y_val, predictions)

print("\n" + "="*50)
print(f"🚀 Final Tuned XGBoost R2 Score: {final_r2 * 100:.2f}%")
print("="*50)

Transforming feature matrices through the traffic pipeline...
Extracting winning parameters from Optuna study...
Fitting final production XGBoost model...
Generating test inferences...

🚀 Final Tuned XGBoost R2 Score: 94.34%


In [9]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn import metrics

def objective(trial):
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'random_state': 42,
        'force_row_wise': True,
        'verbose': -1,
        'n_jobs': -1,  # Utilizes all CPU cores to speed up tuning trials
        
        # Hyperparameter search space
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # Regularization (Lasso/Ridge equivalents) to prevent overfitting
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 15.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True)
    }

    # Use the clean, isolated encoded matrices
    model = lgb.LGBMRegressor(**param)
    model.fit(X_train_optuna, y_train)

    preds = model.predict(X_val_optuna)
    preds = np.clip(preds, a_min=0, a_max=None) # Keep boundaries safe
    
    return metrics.r2_score(y_val, preds)

print("Pre-processing clean, isolated streams for LightGBM tuning...")
# FIX: Encodes data into separate pointers so you can re-run this cell safely anytime
X_train_optuna = X_train.copy()
X_val_optuna = X_val.copy()

print("Starting leak-free LightGBM hyperparameter tuning...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\n--- LIGHTGBM TUNING COMPLETE ---")
print(f"Best Tuned Validation R2 Score: {study.best_value * 100:.2f}%")
print("Best Hyperparameters Found:", study.best_params)

[I 2026-05-29 13:00:16,622] A new study created in memory with name: no-name-8c55e2ca-1edb-462b-9e10-3f0f7148b79c


Pre-processing clean, isolated streams for LightGBM tuning...
Starting leak-free LightGBM hyperparameter tuning...


[I 2026-05-29 13:00:21,438] Trial 0 finished with value: 0.9173152555128319 and parameters: {'n_estimators': 900, 'learning_rate': 0.0630288704821902, 'num_leaves': 68, 'max_depth': 12, 'min_child_samples': 56, 'subsample': 0.838657408122089, 'colsample_bytree': 0.8050661148728044, 'reg_alpha': 0.20364289140180125, 'reg_lambda': 0.10889518726672522}. Best is trial 0 with value: 0.9173152555128319.
[I 2026-05-29 13:00:25,307] Trial 1 finished with value: 0.8420587387198057 and parameters: {'n_estimators': 1200, 'learning_rate': 0.010442494807765022, 'num_leaves': 48, 'max_depth': 5, 'min_child_samples': 100, 'subsample': 0.877003558732196, 'colsample_bytree': 0.9718170195286028, 'reg_alpha': 0.10506740255964094, 'reg_lambda': 0.13928657810480055}. Best is trial 0 with value: 0.9173152555128319.
[I 2026-05-29 13:00:26,444] Trial 2 finished with value: 0.8348208617036745 and parameters: {'n_estimators': 900, 'learning_rate': 0.08785200432802955, 'num_leaves': 53, 'max_depth': 8, 'min_chil


--- LIGHTGBM TUNING COMPLETE ---
Best Tuned Validation R2 Score: 93.95%
Best Hyperparameters Found: {'n_estimators': 1000, 'learning_rate': 0.15642915843582583, 'num_leaves': 69, 'max_depth': 12, 'min_child_samples': 12, 'subsample': 0.7646831393971377, 'colsample_bytree': 0.8126127521091899, 'reg_alpha': 0.1411477892267557, 'reg_lambda': 1.2046380138512}


In [31]:
import lightgbm as lgb
import numpy as np
from sklearn import metrics

# 1. Pre-process clean, isolated streams for final training
# FIX: Passes transformed data to new variable names to prevent 'run-cell-twice' corruption
print("Transforming feature matrices through the traffic pipeline...")

# 2. Extract and combine the best parameters dynamically
print("Extracting winning parameters from LightGBM Optuna study...")
production_params = {
    'objective': 'regression',
    'metric': 'huber',
    
    'force_row_wise': True,
    'verbose': -1,
    'n_jobs': -1,                        # Accelerates training using all CPU threads
    
    # --- AUTOMATIC SPLICE ---
    # Unpacks all optimal parameters found during your LightGBM study search loop
     'n_estimators': 2022,
  'learning_rate': 0.07152358534853247,
  'num_leaves': 100,
  'max_depth': 14,
  'min_child_samples': 5,
  'subsample': 0.8125871599243566,
  'colsample_bytree': 0.9452065188677843,
    
}

# 3. Initialize the production model
model_lgb = lgb.LGBMRegressor(**production_params)

# 4. Train the model on 100% of the training features
print("Fitting final production LightGBM model...")
model_lgb.fit(X_train, y_train)

# 5. Make predictions and safely bound them
print("Generating test inferences...")
predictions = model_lgb.predict(X_val)
predictions = np.clip(predictions, a_min=0, a_max=None) # Keeps floor bounded at zero traffic

# 6. Evaluate final R2 score performance
final_r2 = metrics.r2_score(y_val, predictions)

print("\n" + "="*50)
print(f"🚀 Final Tuned LightGBM R2 Score: {final_r2 * 100:.2f}%")
print("="*50)

Transforming feature matrices through the traffic pipeline...
Extracting winning parameters from LightGBM Optuna study...
Fitting final production LightGBM model...
Generating test inferences...

🚀 Final Tuned LightGBM R2 Score: 94.67%


In [14]:
import optuna
import catboost as cb
import numpy as np
from sklearn import metrics

def objective_cb(trial):
    param = {
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': False,
        'bootstrap_type': 'Bernoulli', 
        'thread_count': -1,            # Utilizes all CPU threads to accelerate execution
        
        # Maxed-Out Dials with Strict Engine Boundaries Allowed
        'iterations': trial.suggest_int('iterations', 400, 3000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.25, log=True),
        
        # FIX: Capped strictly at 10. Avoids 'depth > 16' crashes and prevents massive CPU slowdowns
        'depth': trial.suggest_int('depth', 4, 10), 
        
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 30.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0)
    }

    # Initialize model using current trial parameters
    model = cb.CatBoostRegressor(**param)
    
    # Fit using your clean numeric engineered data
    model.fit(
        X_train_optuna, y_train,
        eval_set=(X_val_optuna, y_val),
        early_stopping_rounds=40,      # Halts early if validation R2 plateaus
        verbose=False
    )

    preds = model.predict(X_val_optuna)
    preds = np.clip(preds, a_min=0, a_max=None) # Ground negative traffic predictions to 0
    
    return metrics.r2_score(y_val, preds)

print("Preparing flat numeric arrays for leak-free CatBoost tuning...")
# Link directly to your flat engineered data splits
X_train_optuna = X_train
X_val_optuna = X_val
y_val = y_val

print("Starting fixed, high-velocity CatBoost hyperparameter tuning...")
study_cb = optuna.create_study(direction='maximize')
# Running 20 trials because deep trees take more computation time
study_cb.optimize(objective_cb, n_trials=20)

print("\n--- CATBOOST TUNING COMPLETE ---")
print(f"Best Tuned Validation R2 Score: {study_cb.best_value * 100:.2f}%")
print("Best Hyperparameters Found:", study_cb.best_params)

[I 2026-05-29 14:04:43,843] A new study created in memory with name: no-name-4a353ee5-0d3f-40d2-8f85-88ddb1a8a40c


Preparing flat numeric arrays for leak-free CatBoost tuning...
Starting fixed, high-velocity CatBoost hyperparameter tuning...


[I 2026-05-29 14:05:03,615] Trial 0 finished with value: 0.9138060372556619 and parameters: {'iterations': 1900, 'learning_rate': 0.09435946400071232, 'depth': 8, 'l2_leaf_reg': 2.3112686364864996, 'subsample': 0.515017194874963}. Best is trial 0 with value: 0.9138060372556619.
[I 2026-05-29 14:05:24,359] Trial 1 finished with value: 0.9058039937649267 and parameters: {'iterations': 2600, 'learning_rate': 0.05349024326205185, 'depth': 7, 'l2_leaf_reg': 21.591512853157745, 'subsample': 0.7430959727333403}. Best is trial 0 with value: 0.9138060372556619.
[I 2026-05-29 14:05:37,864] Trial 2 finished with value: 0.9161636637759177 and parameters: {'iterations': 1800, 'learning_rate': 0.13545107287468428, 'depth': 7, 'l2_leaf_reg': 0.2856630082683813, 'subsample': 0.6153981506083352}. Best is trial 2 with value: 0.9161636637759177.
[I 2026-05-29 14:05:54,348] Trial 3 finished with value: 0.874198017200758 and parameters: {'iterations': 2800, 'learning_rate': 0.035888096119319625, 'depth': 5


--- CATBOOST TUNING COMPLETE ---
Best Tuned Validation R2 Score: 93.02%
Best Hyperparameters Found: {'iterations': 500, 'learning_rate': 0.24311519934323275, 'depth': 10, 'l2_leaf_reg': 0.09818511072345523, 'subsample': 0.999356807009053}


In [50]:
import catboost as cb
import numpy as np
from sklearn import metrics

# 1. Pre-process clean, isolated streams for final training
print("Preparing feature matrices for production training...")
# Since everything is already numeric and pre-encoded, we pass the features directly
X_train_proc = X_train
X_val_proc = X_val

# 2. Extract and combine the best parameters dynamically
print("Extracting winning parameters from CatBoost Optuna study...")
production_params = {
    'loss_function': 'RMSE',
    'random_seed': 42,
    'bootstrap_type': 'Bernoulli',
    'verbose': False,
    'thread_count': -1,                  # Accelerates training using all CPU threads
    
    # --- AUTOMATIC SPLICE ---
    # Unpacks all optimal parameters found during your CatBoost study search loop
    **study_cb.best_params,
    
    # Optional: Safe fallbacks for regularizations if they weren't selected in a short run
    'l2_leaf_reg': study_cb.best_params.get('l2_leaf_reg', 3.0)
}

# 3. Initialize the production model
model_cb = cb.CatBoostRegressor(**production_params)

# 4. Train the model on 100% of the training features
print("Fitting final production CatBoost model...")
model_cb.fit(X_train_proc, y_train)

# 5. Make predictions and safely bound them
print("Generating test inferences...")
predictions = model_cb.predict(X_val_proc)
predictions = np.clip(predictions, a_min=0, a_max=None) # Keeps floor bounded at zero traffic

# 6. Evaluate final R2 score performance
final_r2 = metrics.r2_score(y_val, predictions)

print("\n" + "="*50)
print(f"🚀 Final Tuned CatBoost R2 Score: {final_r2 * 100:.2f}%")
print("="*50)

Preparing feature matrices for production training...
Extracting winning parameters from CatBoost Optuna study...
Fitting final production CatBoost model...
Generating test inferences...

🚀 Final Tuned CatBoost R2 Score: 93.01%


In [ ]:
import optuna
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn import metrics

def objective_rf(trial):
    param = {
        'random_state': 42,
        'n_jobs': -1,
        
        # Maxed-Out Hyperparameter Search Space
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500, step=100),
        'max_depth': trial.suggest_int('max_depth', 8, 40),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.3, 0.9),
        
        # Advanced Ensembling Parameters
        'bootstrap': True,
        'ccp_alpha': trial.suggest_float('ccp_alpha', 1e-5, 1e-1, log=True) # Cost-complexity pruning
    }

    model = RandomForestRegressor(**param)
    model.fit(X_train_optuna, y_train)

    preds = model.predict(X_val_optuna)
    preds = np.clip(preds, a_min=0, a_max=None)
    
    return metrics.r2_score(y_val, preds)

print("Preparing isolated matrices for extreme Random Forest tuning...")
X_train_optuna = traffic_pipeline.fit_transform(X_train)
X_val_optuna = traffic_pipeline.transform(X_test)
y_val = y_test

print("Starting extreme Random Forest hyperparameter tuning...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=25)

print("\n🏆 Best Random Forest Validation R2 Score: {study_rf.best_value * 100:.2f}%")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn import metrics

# 1. Pre-process clean, isolated streams for final training
# FIX: Passes transformed data to new variable names to prevent 'run-cell-twice' corruption
print("Transforming feature matrices through the traffic pipeline...")
X_train_proc = traffic_pipeline.fit_transform(X_train)
X_test_proc = traffic_pipeline.transform(X_test)

# 2. Extract and combine the best parameters dynamically
print("Extracting winning parameters from Random Forest Optuna study...")
production_params = {
    'random_state': 42,
    'n_jobs': -1,                        # Forces execution across all available CPU cores
    
    # --- AUTOMATIC SPLICE ---
    # Unpacks all optimal parameters found during your maxed-out study_rf search loop
    **study_rf.best_params
}

# 3. Initialize the production model
model_rf = RandomForestRegressor(**production_params)

# 4. Train the model on 100% of the training features
print("Fitting final production Random Forest model...")
model_rf.fit(X_train_proc, y_train)

# 5. Make predictions and safely bound them
print("Generating test inferences...")
rf_predictions = model_rf.predict(X_test_proc)
rf_predictions = np.clip(rf_predictions, a_min=0, a_max=None) # Ground negative predictions to 0

# 6. Evaluate final R2 score performance
rf_r2 = metrics.r2_score(y_test, rf_predictions)
rf_hackathon_score = max(0, 100 * rf_r2)

print("\n" + "="*50)
print(f"🚀 Final Tuned Random Forest R2 Score: {rf_r2 * 100:.2f}%")
print(f"🏆 Final Hackathon Score Benchmark  : {rf_hackathon_score:.2f}")
print("="*50)

In [59]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

# 1. PASTE YOUR TUNED PARAMETERS HERE
xgb_params = {
    'n_estimators': 442,
    'learning_rate': 0.04254848876707133,
    'max_depth': 14,
    'subsample': 0.7392448086836987,
    'colsample_bytree': 0.9999598584570065,
    'min_child_weight': 6,     
    'device': 'cuda',
    'tree_method': 'hist',
    'objective': 'reg:squarederror', 
    'random_state': 42
}

lgb_params = {
    'n_estimators': 1022,
    'learning_rate': 0.07152358534853247,
    'num_leaves': 98,
    'max_depth': 14,
    'min_child_samples': 5,
    'subsample': 0.8125871599243566,
    'colsample_bytree': 0.9452065188677843,   
    'objective': 'regression',
    'metric': 'rmse',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

cb_params = {
    'iterations': 806,
    'learning_rate': 0.2053121405817385,
    'depth': 10,
    'l2_leaf_reg': 4.853841536677037,
    'subsample': 0.9975941401259907,
    'loss_function': 'RMSE', 
    'random_seed': 42, 
    'verbose': False, 
    'bootstrap_type': 'Bernoulli',
}

rf_params = {
    'n_estimators': 489, 
    'max_depth': 28, 
    'min_samples_split': 10, 
    'min_samples_leaf': 1, 
    'max_features': 0.7942355592662035,
    'random_state': 42,
    'n_jobs': -1
}

# 2. Initialize all four models
print("Initializing all 4 optimized models...")
model_xgb = xgb.XGBRegressor(**xgb_params)
model_lgb = lgb.LGBMRegressor(**lgb_params)
model_cb = cb.CatBoostRegressor(**cb_params)
model_rf = RandomForestRegressor(**rf_params)

Initializing all 4 optimized models...


In [60]:
model_xgb.fit(X_train, y_train)
model_lgb.fit(X_train, y_train)
model_cb.fit(X_train, y_train)

CatBoostRegressor(bootstrap_type='Bernoulli', depth=10, iterations=806, l2_leaf_reg=4.853841536677037, learning_rate=0.2053121405817385, loss_function='RMSE', random_seed=42, subsample=0.9975941401259907, verbose=False)

In [55]:
import optuna
import numpy as np
from sklearn import metrics

print("Preparing clean, independent data streams to prevent prediction crashes...")

# STREAM A: One-Hot Encoded data matrix used by LightGBM and XGBoost
X_test_encoded =X_val.copy() # FIX: Creates a fresh copy to prevent Jupyter state bugs
# Assuming traffic_pipeline is your pre-fitted ColumnTransformer or Pipeline that encodes the data

# STREAM B: Native string DataFrame used exclusively by CatBoost
X_test_native_strings = X_val.copy()
categorical_cols = ['Weather', 'RoadType', 'Landmarks']
for col in categorical_cols:
    if col in X_test_native_strings.columns:
        X_test_native_strings[col] = X_test_native_strings[col].astype(str)

# 1. Generate static validation predictions using the correct stream for each architecture
print("Gathering pre-computed baseline model predictions...")
pred_lgb = model_lgb.predict(X_test_encoded)
pred_xgb = model_xgb.predict(X_test_encoded)
pred_cb  = model_cb.predict(X_test_native_strings) # FIX: Clean, explicit stream bypasses Jupyter state bugs

def objective_weights(trial):
    # 2. Let Optuna search for the best weight distributions between 0.0 and 1.0
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_cb  = trial.suggest_float('w_cb',  0.0, 1.0)

    # 3. Normalize the weights so they always add up exactly to 1.0 (100%)
    total_weight = w_lgb + w_xgb + w_cb
    if total_weight == 0:
        return 0.0 # Prevent division by zero errors

    weight_xgb = w_xgb / total_weight
    weight_lgb = w_lgb / total_weight
    weight_cb  = w_cb  / total_weight

    # 4. Blend the predictions using the normalized weights
    blended_preds = (
        (weight_lgb * pred_lgb) + 
        (weight_xgb * pred_xgb) + 
        (weight_cb  * pred_cb)
    )
    
    # Floor boundary protection at zero traffic
    blended_preds = np.clip(blended_preds, a_min=0, a_max=None)

    # 5. Calculate the R2 score for this trial blend configuration
    r2 = metrics.r2_score(y_val, blended_preds)
    return r2

# 6. Run 1000 lightning-fast meta-trials
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_weights = optuna.create_study(direction='maximize')

print("Hunting for perfect meta-ensemble blending weights...")
study_weights.optimize(objective_weights, n_trials=1000)

# 7. Extract and normalize the absolute best weights found
best_w = study_weights.best_params
total_best_w = best_w['w_lgb'] + best_w['w_xgb'] + best_w['w_cb']

final_w_xgb = best_w['w_xgb'] / total_best_w
final_w_lgb = best_w['w_lgb'] / total_best_w
final_w_cb  = best_w['w_cb']  / total_best_w

print("\n" + "="*50)
print("🏆 3-WAY META-OPTIMIZATION COMPLETE")
print("="*50)
print(f"Maximized Ensemble Blended R2 : {study_weights.best_value * 100:.2f}%")
print(f"Optimal XGBoost Contribution  : {final_w_xgb * 100:.2f}%")
print(f"Optimal LightGBM Contribution : {final_w_lgb * 100:.2f}%")
print(f"Optimal CatBoost Contribution : {final_w_cb * 100:.2f}%")
print("="*50)

Preparing clean, independent data streams to prevent prediction crashes...
Gathering pre-computed baseline model predictions...


c:\Users\Pulkit\Banglore_traffic_prediction\venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [15:09:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Hunting for perfect meta-ensemble blending weights...

🏆 3-WAY META-OPTIMIZATION COMPLETE
Maximized Ensemble Blended R2 : 94.54%
Optimal XGBoost Contribution  : 28.41%
Optimal LightGBM Contribution : 71.59%
Optimal CatBoost Contribution : 0.00%


In [61]:
import optuna
import numpy as np
from sklearn import metrics

print("Generating validation predictions...")
pred_xgb = model_xgb.predict(X_val) # Use your validation-trained XGBoost
pred_lgb = model_lgb.predict(X_val) # Use your validation-trained LightGBM
pred_cb = model_cb.predict(X_val)   # Use your validation-trained CatBoost
 # Use your validation-trained Random Forest

def objective_weights(trial):
    # 2. Let Optuna guess weights between 0 and 1 for all 4 models
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_cb = trial.suggest_float('w_cb', 0.0, 1.0)
   
    
    # 3. Normalize the weights so they always add up exactly to 1.0 (100%)
    total_weight = w_xgb + w_lgb + w_cb 
    if total_weight == 0: 
        return 0 # Prevent division by zero
    
    weight_xgb = w_xgb / total_weight
    weight_lgb = w_lgb / total_weight
    weight_cb = w_cb / total_weight
  
    
    # 4. Blend the predictions using the guessed weights
    blended_preds = (weight_xgb * pred_xgb) + \
                    (weight_lgb * pred_lgb) + \
                    (weight_cb * pred_cb) 
    
    # 5. Calculate the R2 Score of this specific blend
    r2 = metrics.r2_score(y_val, blended_preds)
    return r2

# 6. Run 1000 lightning-fast trials (takes just a few seconds)
study_weights = optuna.create_study(direction='maximize')
print("Hunting for the perfect 4-way ensemble weights...")
study_weights.optimize(objective_weights, n_trials=3000)

# 7. Extract and normalize the absolute best weights found
best_w = study_weights.best_params
total_best_w = best_w['w_xgb'] + best_w['w_lgb'] + best_w['w_cb'] 

final_w_xgb = best_w['w_xgb'] / total_best_w
final_w_lgb = best_w['w_lgb'] / total_best_w
final_w_cb = best_w['w_cb'] / total_best_w


print("\n--- PERFECT WEIGHTS FOUND ---")
print(f"Maximized Validation R2: {max(0, 100 * study_weights.best_value):.2f}")
print(f"XGBoost Weight:       {final_w_xgb:.4f}")
print(f"LightGBM Weight:      {final_w_lgb:.4f}")
print(f"CatBoost Weight:      {final_w_cb:.4f}")


Generating validation predictions...
Hunting for the perfect 4-way ensemble weights...

--- PERFECT WEIGHTS FOUND ---
Maximized Validation R2: 94.54
XGBoost Weight:       0.2877
LightGBM Weight:      0.7123
CatBoost Weight:      0.0000


In [62]:
import pandas as pd
import numpy as np

print("Loading the separate engineered test dataset...")
# 1. Load your standalone test file directly
df_test = pd.read_csv('test_engineered.csv')

# 2. Assign the columns directly as your features
X_test_final = df_test

print("Generating predictions directly from your numeric models...")
# 3. Predict directly using the exact same input matrix for all 3 models
test_pred_xgb = model_xgb.predict(X_test_final)
test_pred_lgb = model_lgb.predict(X_test_final)
test_pred_cb  = model_cb.predict(X_test_final)

print("Applying your dynamically calculated Optuna weights to synthesize the blend...")
# 4. Blend using your exact calculated final weights
final_predictions = (
    (final_w_xgb * test_pred_xgb) + 
    (final_w_lgb * test_pred_lgb) + 
    (final_w_cb * test_pred_cb)
)

# 5. Guard against impossible negative traffic values
final_predictions = np.clip(final_predictions, a_min=0, a_max=None)

print("Building basic submission dataframe...")
# 6. Map predictions directly back to the index of your test file
submission_df = pd.DataFrame({
    'Index': df_test.index, 
    'demand': final_predictions
})

# 7. Save the clean submission file to disk
submission_df.to_csv('perfect_alignment_submission6.csv', index=False)

print("\n" + "="*60)
print("🎉 SUCCESS: Submission file generated using optimized weights!")
print("Filename          : perfect_alignment_submission6 .csv")
print(f"Total Rows Aligned: {len(submission_df)}")
print("="*60)

Loading the separate engineered test dataset...
Generating predictions directly from your numeric models...
Applying your dynamically calculated Optuna weights to synthesize the blend...
Building basic submission dataframe...

🎉 SUCCESS: Submission file generated using optimized weights!
Filename          : perfect_alignment_submission6 .csv
Total Rows Aligned: 41778
